In [ ]:
# This is the test case provided by Rory in the follow gitgist.
# This code has been modified to get it to run.

# https://gist.github.com/f0uriest/b687cdf1da08db155b83ee8d16a0bbd6 

In [ ]:
import numpy as np
import yancc
from yancc.field import Field
from yancc.velocity_grids import MaxwellSpeedGrid, UniformPitchAngleGrid
from yancc.species import LocalMaxwellian, GlobalMaxwellian
from yancc.solve import solve_dke
from yancc.misc import normalize_fluxes_sfincs


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, force=True)

In [ ]:
nx = 11 # resolution in x/ speed coordinate
na = 33 # resolution in  pitch angle coordinate
nt = 17 # resolution in theta / poloidal angle
nz = 65 # resolution in zeta / toroidal angle

speedgrid = MaxwellSpeedGrid(nx)
pitchgrid = UniformPitchAngleGrid(na)

vmec_path = '/u/npablant/data/w7x/vmec/w7x_ref_172/wout.nc'
rho = 0.5 # surface label
# field = Field.from_desc(eq, rho, nt, nz)
field = Field.from_vmec(vmec_path, np.sqrt(rho), nt, nz)
# field = Field.from_booz_xform(booz_path, np.sqrt(rho), nt, nz, cutoff=1e-6) # for stellopt or simsopt booz_xform files
# field = Field.from_ipp_bc(bc_path, np.sqrt(rho), nt, nz, cutoff=1e-6) # for IPP booz_xform files

# note all profiles etc should use radial coordinate rho = sqrt(normalized toroidal flux) = r/a

Erho = -2.3e3 # radial electric field Erho = -∂Φ /∂ρ, in Volts

species = [
    GlobalMaxwellian(
        yancc.species.Hydrogen,
        # give full profiles of T, n vs rho in eV and m^-3, 
        temperature=lambda r: 3.0e3 * (1 - r**2),
        density=lambda r: 2e20 * (1 - r**4),
        # then get the value on a single surface
    ).localize(field.rho),
    
    GlobalMaxwellian(
        yancc.species.Electron,
        # give full profiles of T, n vs rho in eV and m^-3, 
        temperature=lambda r: 3.0e3 * (1 - r**2),
        density=lambda r: 2e20 * (1 - r**4),
        # then get the value on a single surface
    ).localize(field.rho)
]

# or create LocalMaxwellian on a single surface, just giving T, n and gradients wrt rho 
#species = [
#    LocalMaxwellian(
#        # can just give mass and charge in units of proton mass and elementary charge
#        yancc.species.Species(1,1), 
#        temperature=1.5e3, 
#        density=1e20, 
#        dTdrho=-1.5e3, 
#        dndrho=-5e19),
#    ]
    
f, rhs, fluxes, stats  = solve_dke(field, pitchgrid, speedgrid, species, Erho, print_every=10)
# f is the distribution function
# rhs is the drive from leading order Maxwellian
# fluxes is a dict of heat,particle,momentum flux etc, in SI units
# stats contains info about number of iterations, residual etc.


sfincs_fluxes = normalize_fluxes_sfincs(fluxes, field, pitchgrid, speedgrid, species)
# sfincs_fluxes is a dict with the same data as fluxes, but with the names that sfincs
# uses and normalized in the same way for comparison.

In [ ]:
fluxes

In [ ]:
sfincs_fluxes